In [35]:
from cgra import *
from kernels import *
from sat_to_csv import *
import random

In [36]:
kernel_name = "conv2d"
version = "_meth"

In [37]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
BLOCK_SIZE = 4
# Adress
first_addr = 20000

In [38]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [39]:
# Data
def configMemory(image_data, filter_data, rowsIm, colsIm):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # 4*nCols               -16*colBlocks           16*nCols            nItLoop1
    # nItLoop2              4*nCols                 -16*colBlocks       16*nCols
    # 16*nCols              -                       4*nCols             -16*colBlocks
    # -16*colBlocks         16*nCols                -                   4*nCols
    # ----------------------
    # &Im[0]                &out[1*nCols + 2]      &Filter[0]              -
    # -                     &Im[1*nCols + 1]       &out[2*nCols + 3]       -
    # -                     -                      &Im[2*nCols + 2]     &out[3*nCols + 4]      
    # &out[4*nCols + 1]     -                      -                    &Im[3*nCols + 3]
    nRowsBlocks = int((rowsIm-2)/BLOCK_SIZE)
    nColsBlocks = int((colsIm-2)/BLOCK_SIZE)
    nItLoop1 = nRowsBlocks
    nItLoop2 = nColsBlocks


    first_addr_im = first_addr
    first_addr_filter = first_addr_im + rowsIm*colsIm*4
    first_addr_out = first_addr_filter + 9*4

    config_vals_col0 = [4*colsIm, nItLoop2, 16*colsIm, -16*nColsBlocks, first_addr_im, first_addr_out + (4*colsIm + 1)*4]
    config_vals_col1 = [-16*nColsBlocks, 4*colsIm, 16*colsIm, first_addr_out + (colsIm + 2)*4, first_addr_im + (colsIm + 1)*4]
    config_vals_col2 = [16*colsIm, -16*nColsBlocks, 4*colsIm, first_addr_filter, first_addr_out + (2*colsIm + 3)*4, first_addr_im + (2*colsIm + 2)*4]
    config_vals_col3 = [nItLoop1, 16*colsIm, -16*nColsBlocks, 4*colsIm, first_addr_out + (3*colsIm + 4)*4, first_addr_im + (3*colsIm + 3)*4]
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_im, image_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_filter, filter_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_out, [0 for _ in range(rowsIm*colsIm)], version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [40]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [41]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [42]:
def conv2d_cpu(image_data, filter_data, rowsIm, colsIm):
    expected_res = [0 for _ in range(rowsIm*colsIm)]
    for i in range(1, rowsIm - 1):
        for j in range(1, colsIm - 1):
            acc = 0
            for fi in range(3):
                for fj in range(3):
                    im_i = i + fi - 1
                    im_j = j + fj - 1
                    im_index = im_i * colsIm + im_j
                    filt_index = fi * 3 + fj
                    acc += image_data[im_index] * filter_data[filt_index]
            expected_res[i * colsIm + j] = acc
    return expected_res

In [43]:
# Test dimensions (4xXx4)
rowsIm = 6
colsIm = 6
image_data = [random.randint(-20, 20) for _ in range(0, rowsIm * colsIm)]
#filter_data = [random.randint(-10, 10) for _ in range(0, 9)]
filter_data = list(range(0,9))
load_addrs = configMemory(image_data, filter_data, rowsIm, colsIm)

In [44]:
runKernel(load_addrs, max_it=200000)

Instr =  0 ( 0 )
[  24,  -16,   96,    1]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[   1,   24,  -16,   96]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[  96,    3,   24,  -16]    [LWD R1  4, SADD R1  ZERO  3, LWD R1  4, LWD R1  4]    
[ -16,   96,    3,   24]    [LWD R1  4, LWD R1  4, SADD R1  ZERO  3, LWD R1  4]    
-------
Instr =  1 ( 1 )
[20000, 20212, 20144,    0]    [LWD R2  4, LWD R2  4, LWD R2  4, SADD R2  ZERO  ZERO]    
[   0, 20028, 20240,   96]    [SADD R2  ZERO  ZERO, LWD R2  4, LWD R2  4, NOP ]    
[  96,    0, 20056, 20268]    [NOP , SADD R2  ZERO  ZERO, LWD R2  4, LWD R2  4]    
[20280,   96,    0, 20084]    [LWD R2  4, NOP , SADD R2  ZERO  ZERO, LWD R2  4]    
-------
Instr =  2 ( 2 )
[20000, 20212, 20144,   29]    [NOP , NOP , NOP , BGE R2  R1  29]    
[   0, 20028, 20240,   96]    [NOP , NOP , NOP , NOP ]    
[  96,    0, 20056, 20268]    [NOP , NOP , NOP , NOP ]    
[20280,   96,    0, 20084]    [NOP , NOP , NOP , NOP ]    
-------
Instr =  3 ( 3

In [45]:
# Get result from CGRA
first_addr_C = first_addr + rowsIm * colsIm * 4 + 9*4
result = getResult(first_addr_C, first_addr_C + rowsIm * colsIm * 4 + 9*4, rowsIm, colsIm)

# Get cpu output
expected_res = conv2d_cpu(image_data, filter_data, rowsIm, colsIm)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("Image: ")
    printAsMatrix(image_data, rowsIm, colsIm)
    print("Filter: ")
    printAsMatrix(filter_data, 3, 3)
    print("Expected: ")
    printAsMatrix(expected_res, rowsIm, colsIm)
    print("CGRA: ")
    printAsMatrix(result, rowsIm, colsIm)
else:
    print("OK")



Err: 16
Image: 
[-9, 17, 11, 6, 1, 20]
[1, 10, 20, -11, 8, 11]
[11, 16, 10, -13, -5, 4]
[-18, -10, 10, -19, -14, 5]
[-14, -4, 7, -19, 20, -12]
[6, 15, 15, 10, 3, 4]
Filter: 
[2, 9, -6]
[-6, -6, -4]
[10, -2, -2]
Expected: 
[0, 0, 0, 0, 0, 0]
[0, -19, 127, 120, -253, 0]
[0, -410, 80, 97, -96, 0]
[0, 88, 260, 111, -123, 0]
[0, -106, 342, 49, -66, 0]
[0, 0, 0, 0, 0, 0]
CGRA: 
[0, 0, 0, 0, 0, 0]
[0, 69, 97, 70, -99, 0]
[0, -28, 266, -107, -16, 0]
[0, 106, 200, -67, -95, 0]
[0, -186, 184, -67, -194, 0]
[0, 0, 0, 0, 0, 0]
